
# Laboratorio avanzado (en español): CrewAI con tools reales + búsqueda web (Serper) + RAG local

Este notebook está listo para **Google Colab** y extiende el laboratorio básico con:

- **Tools reales**
- **Búsqueda web** usando **Serper API**
- **RAG local sencillo** usando archivos del sistema
- **Crew multiagente** con planificación, investigación, redacción y edición

---

## Objetivos
Al finalizar este laboratorio podrás:

1. Crear agentes con CrewAI.
2. Integrar **herramientas reales** de búsqueda y lectura de archivos.
3. Simular un flujo **RAG** con documentos locales.
4. Combinar:
   - contexto dinámico de la web
   - contexto estructurado local
   - redacción y edición final

---

## Caso de estudio
Generar un artículo en español sobre:

> **Sistemas multiagente en inteligencia artificial**

usando:
- investigación web actualizada
- consulta de documentos locales
- redacción estructurada
- edición final


## 1. Instalación de librerías

In [ ]:

!pip -q install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29



## 2. Configuración de credenciales

Este laboratorio usa:

- `OPENAI_API_KEY`
- `SERPER_API_KEY`

> Serper permite hacer búsquedas web sobre Google desde herramientas de CrewAI.


In [ ]:
import os
import warnings
from getpass import getpass

warnings.filterwarnings("ignore")

os.environ["OPENAI_API_KEY"] = 'sk-xxx'
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = 'xxx'
# Puedes cambiar el modelo si deseas
#os.environ["OPENAI_MODEL_NAME"] = "gpt-4o-mini"

print("Modelo configurado:", os.environ["OPENAI_MODEL_NAME"])
print("SERPER_API_KEY cargada:", "Sí" if os.environ.get("SERPER_API_KEY") else "No")



## 3. Importaciones principales


In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool, DirectoryReadTool, FileReadTool
from IPython.display import Markdown, display



## 4. Construcción de una base documental local (RAG sencillo)

Para mantener este notebook auto-contenido, vamos a crear una carpeta local con documentos breves.  
Estos archivos actuarán como una **fuente de conocimiento local** que los agentes podrán consultar.

> Esto no es un vector store completo, pero sí es una forma simple y pedagógica de introducir una lógica tipo RAG.


In [ ]:

import os

os.makedirs("base_conocimiento", exist_ok=True)

doc_1 = '''
Sistemas multiagente (MAS)
Un sistema multiagente es un conjunto de agentes que cooperan para resolver un problema común.
Cada agente puede tener un rol especializado, herramientas y objetivos específicos.
Los MAS son útiles cuando un problema puede dividirse en subtareas complementarias.
'''

doc_2 = '''
CrewAI
CrewAI es un framework open source para construir sistemas multiagente.
Se apoya en tres conceptos principales: agentes, tareas y crews.
Permite orquestar agentes que colaboran secuencialmente, jerárquicamente o con delegación.
'''

doc_3 = '''
Arquitectura de agentes
Un agente puede combinar razonamiento, uso de herramientas, memoria y colaboración.
En sistemas avanzados, algunos agentes investigan, otros sintetizan información y otros validan resultados.
Esto permite diseñar pipelines cognitivos más robustos que un solo LLM aislado.
'''

with open("base_conocimiento/mas.txt", "w", encoding="utf-8") as f:
    f.write(doc_1)

with open("base_conocimiento/crewai.txt", "w", encoding="utf-8") as f:
    f.write(doc_2)

with open("base_conocimiento/arquitectura_agentes.txt", "w", encoding="utf-8") as f:
    f.write(doc_3)

print("Base documental creada en ./base_conocimiento")
print(os.listdir("base_conocimiento"))



## 5. Definición de tools reales

Usaremos tres tools:

1. **SerperDevTool** → búsqueda web actualizada
2. **DirectoryReadTool** → lectura de una carpeta
3. **FileReadTool** → lectura de archivos concretos

Estas tools permiten combinar:
- conocimiento **externo y dinámico** (web)
- conocimiento **interno y persistente** (documentos locales)


In [ ]:

herramienta_web = SerperDevTool()

herramienta_directorio = DirectoryReadTool(directory="base_conocimiento")
herramienta_archivo = FileReadTool()



## 6. Definición de agentes

Tendremos cuatro agentes:

1. **Investigador web**
2. **Investigador documental**
3. **Redactor**
4. **Editor**


### 6.1 Investigador web

In [ ]:

investigador_web = Agent(
    role="Investigador Web",
    goal="Buscar información reciente y relevante en internet sobre el tema: {tema}",
    backstory=(
        "Eres un investigador especializado en recopilación de información reciente. "
        "Tu trabajo es buscar tendencias, conceptos, noticias y desarrollos actuales sobre {tema}. "
        "Debes sintetizar información útil y pertinente para apoyar la redacción del artículo."
    ),
    tools=[herramienta_web],
    allow_delegation=False,
    verbose=True
)


### 6.2 Investigador documental (RAG local)

In [ ]:

investigador_documental = Agent(
    role="Investigador Documental",
    goal="Recuperar conocimiento estructurado desde la base documental local sobre el tema: {tema}",
    backstory=(
        "Eres un analista documental. "
        "Tu función es revisar la base de conocimiento local, identificar documentos relevantes "
        "y extraer conceptos clave, definiciones y relaciones útiles para construir el artículo."
    ),
    tools=[herramienta_directorio, herramienta_archivo],
    allow_delegation=False,
    verbose=True
)


### 6.3 Redactor

In [ ]:

redactor = Agent(
    role="Redactor Técnico",
    goal="Escribir un artículo sólido, claro y bien estructurado sobre el tema: {tema}",
    backstory=(
        "Eres un redactor técnico y académico. "
        "Tu trabajo consiste en combinar la investigación web y la evidencia documental local "
        "para escribir un artículo en español, riguroso, coherente y útil para una audiencia universitaria."
    ),
    allow_delegation=False,
    verbose=True
)


### 6.4 Editor

In [ ]:

editor = Agent(
    role="Editor Académico",
    goal="Revisar y mejorar el artículo final para dejarlo listo para publicación",
    backstory=(
        "Eres un editor con experiencia en escritura académica y técnica. "
        "Debes corregir estilo, claridad, cohesión y tono, asegurando una versión final bien escrita."
    ),
    allow_delegation=False,
    verbose=True
)



## 7. Definición de tareas

Cada tarea tiene:
- una descripción clara
- una salida esperada
- un agente responsable


### 7.1 Tarea: investigación web

In [ ]:

tarea_web = Task(
    description=(
        "Realiza una investigación web sobre {tema}.\n"
        "1. Busca definiciones actuales y desarrollos recientes.\n"
        "2. Identifica tendencias, casos de uso y conceptos relevantes.\n"
        "3. Resume la información en español de manera clara y útil para redacción posterior."
    ),
    expected_output=(
        "Un informe breve en español con hallazgos web recientes sobre el tema, "
        "incluyendo conceptos clave, tendencias y observaciones relevantes."
    ),
    agent=investigador_web
)


### 7.2 Tarea: investigación documental (RAG local)

In [ ]:

tarea_rag = Task(
    description=(
        "Consulta la base de conocimiento local sobre {tema}.\n"
        "1. Revisa el contenido del directorio base_conocimiento.\n"
        "2. Identifica qué archivos son relevantes.\n"
        "3. Extrae definiciones, conceptos y relaciones importantes.\n"
        "4. Resume en español los elementos documentales más útiles para la redacción."
    ),
    expected_output=(
        "Un informe documental en español con conceptos clave extraídos de los archivos locales "
        "y una síntesis útil para elaborar el artículo."
    ),
    agent=investigador_documental
)


### 7.3 Tarea: redacción del artículo

In [ ]:

tarea_redaccion = Task(
    description=(
        "Usa la investigación web y la investigación documental para escribir un artículo sobre {tema}.\n"
        "El artículo debe:\n"
        "- Estar en español\n"
        "- Tener introducción, desarrollo y conclusión\n"
        "- Integrar aportes de ambas fuentes (web + base local)\n"
        "- Mantener tono profesional, pedagógico y técnico\n"
        "- Usar subtítulos claros\n"
    ),
    expected_output=(
        "Un artículo completo en español, en formato Markdown, con estructura clara y buena integración "
        "de la evidencia obtenida por los agentes investigadores."
    ),
    agent=redactor
)


### 7.4 Tarea: edición final

In [ ]:

tarea_edicion = Task(
    description=(
        "Revisa el artículo generado y mejora su calidad final.\n"
        "- Corrige errores gramaticales\n"
        "- Mejora cohesión, claridad y estilo\n"
        "- Conserva formato Markdown\n"
        "- Asegura una versión final lista para presentar o publicar"
    ),
    expected_output=(
        "Un artículo final en español, revisado y optimizado, en formato Markdown."
    ),
    agent=editor
)



## 8. Construcción del crew

El flujo de este laboratorio será:

1. Investigación web
2. Investigación documental local (RAG)
3. Redacción
4. Edición


In [ ]:

equipo = Crew(
    agents=[investigador_web, investigador_documental, redactor, editor],
    tasks=[tarea_web, tarea_rag, tarea_redaccion, tarea_edicion],
    verbose=2
)



## 9. Ejecución

Puedes cambiar el valor de `tema` libremente.


In [ ]:

tema = "Sistemas multiagente en inteligencia artificial"

resultado = equipo.kickoff(inputs={"tema": tema})


## 10. Visualización del resultado

In [ ]:

display(Markdown(resultado))



## 11. Probar con otro tema

Prueba, por ejemplo:

- `Agentes IA en educación superior`
- `RAG y sistemas multiagente`
- `Automatización inteligente con CrewAI`


In [ ]:

nuevo_tema = "RAG y sistemas multiagente"

resultado_2 = equipo.kickoff(inputs={"tema": nuevo_tema})
display(Markdown(resultado_2))



## 12. ¿Dónde está el componente RAG aquí?

En este laboratorio, el componente tipo RAG aparece cuando:

- el agente **investigador documental** consulta una base documental local,
- recupera fragmentos/archivos relevantes,
- y usa esa recuperación como contexto para la redacción posterior.

### Esquema conceptual

**Pregunta / tema**  
→ **Recuperación de conocimiento local**  
→ **Síntesis del contexto**  
→ **Generación del artículo**

---

### Diferencia frente a un RAG completo con embeddings
Aquí usamos una versión **pedagógica y simple**:

- recuperación basada en archivos
- sin vector store
- sin embeddings explícitos

En una versión más avanzada podrías incorporar:

- FAISS / Chroma
- embeddings
- chunking
- reranking



## 13. Extensiones sugeridas para estudiantes

### Extensión 1
Agregar un **agente verificador de fuentes**.

### Extensión 2
Convertir el flujo en una arquitectura **jerárquica** con un agente manager.

### Extensión 3
Reemplazar el RAG simple por:
- embeddings
- vector store
- búsqueda semántica

### Extensión 4
Generar además:
- resumen ejecutivo
- presentación en viñetas
- borrador de clase o conferencia



## 14. Reflexión final

Este laboratorio integra cuatro ideas poderosas:

1. **CrewAI como orquestador**
2. **Tools reales para actuar fuera del LLM**
3. **RAG local como memoria externa/documental**
4. **Especialización de agentes para mejorar calidad**

---

### Mensaje clave
Un sistema multiagente bien diseñado no solo genera texto:
**investiga, recupera, sintetiza, redacta y mejora**.
